# Overlap check: do the test prompts occur in the official training data?

**No GPU needed** (CPU session, Internet **On**, Secret `HF_TOKEN`). Attach your best submission zip (the 0.8338 one) as a dataset; it is found automatically (or set `BASE_ZIP`).

1. Streams all four Phase-1 training files from Hugging Face (nothing is stored).
2. For each of the 1,000 test prompts, searches the corpus for its **last ~120 characters** (exact match after whitespace normalisation), and also for a passage from the **middle** of the prompt (diagnostic: is the source document in the training data at all?).
3. Prints the hit rate. Where the prompt end is found, the 128 words that follow it in the training document are used as the continuation; every other prompt keeps the model's output from `BASE_ZIP`.



In [ ]:
!pip -q install pyahocorasick ijson

In [ ]:
import os, json, zipfile, time, glob
from collections import Counter
from multiprocessing import Pool
import ahocorasick, ijson
from huggingface_hub import login, HfApi, HfFileSystem, hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN")); print("logged in (Kaggle secret)")
except Exception:
    import getpass
    login(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token: ")); print("logged in")
print("ijson backend:", ijson.backend)

REPO = "Exploration-Lab/CS-781-Capstone"
SOURCES = ("acts", "high_courts", "supreme_court", "tribunals")
WORDS = 128
TAIL_CHARS = 120          # length of the prompt-end anchor
MIN_WORDS = 64            # a corpus continuation shorter than this is ignored
BASE_ZIP = None           # e.g. "/kaggle/input/<dataset>/submission_arch4big.zip"; None = search /kaggle/input
OUT = "/kaggle/working"

files = [f for f in HfApi().list_repo_files(REPO, repo_type="dataset") if f.startswith("Phase_1")]
TRAIN = {s: next(f for f in files if "/Train/" in f and s in f.lower()) for s in SOURCES}
TEST_FILE = next(f for f in files if "/Test/" in f and f.endswith(".json"))
print(TRAIN, TEST_FILE)

In [ ]:
# test prompts + the model's predictions to fall back on
def pick(v):
    return v if isinstance(v, str) else next((v[k] for k in ("text", "content", "prompt", "body") if isinstance(v.get(k), str)), "")
test = {k: pick(v) for k, v in json.load(open(hf_hub_download(REPO, TEST_FILE, repo_type="dataset"), encoding="utf-8")).items()}
norm = lambda t: " ".join(t.split())

if BASE_ZIP is None:
    cands = glob.glob("/kaggle/input/**/*.zip", recursive=True) + glob.glob("/kaggle/input/**/prediction.json", recursive=True)
    assert cands, "attach your best submission zip (Add Input) or set BASE_ZIP"
    print("found:", cands); BASE_ZIP = cands[0]
if BASE_ZIP.endswith(".zip"):
    with zipfile.ZipFile(BASE_ZIP) as z: base = json.loads(z.read("prediction.json"))
else:
    base = json.load(open(BASE_ZIP, encoding="utf-8"))
assert set(base) == set(test), "base predictions do not match the test ids"
print(f"{len(test)} test prompts | base predictions from {BASE_ZIP}")

# anchors: prompt end (gives the continuation) and a passage from the middle (diagnostic)
PATTERNS = {}
for k, t in test.items():
    n = norm(t)
    tail = n[-TAIL_CHARS:]; tail = tail[tail.find(" ") + 1:]          # start on a word boundary
    w = n.split(); mid = " ".join(w[len(w) // 2: len(w) // 2 + 20])
    PATTERNS.setdefault(tail, []).append(("end", k)); PATTERNS.setdefault(mid, []).append(("mid", k))
print(f"{len(PATTERNS)} anchors")

In [ ]:
# stream each source in its own process and look for every anchor in every document
def scan(src):
    A = ahocorasick.Automaton()
    for p, owners in PATTERNS.items(): A.add_word(p, (p, owners))
    A.make_automaton()
    fs, t0 = HfFileSystem(), time.time()
    path = f"datasets/{REPO}/{TRAIN[src]}"
    with fs.open(path, "rb", block_size=16 << 20) as f: head = f.read(64).lstrip()[:1]
    for attempt in range(3):
        cont, mid, docs, chars = {}, set(), 0, 0
        try:
            with fs.open(path, "rb", block_size=16 << 20) as f:
                items = ijson.kvitems(f, "") if head == b"{" else ijson.items(f, "item")
                for item in items:
                    v = item[1] if head == b"{" else item
                    t = v if isinstance(v, str) else (v.get("text") if isinstance(v, dict) else None)
                    if not isinstance(t, str): continue
                    n = norm(t); docs += 1; chars += len(n)
                    for end, (p, owners) in A.iter(n):
                        for kind, k in owners:
                            if kind == "mid": mid.add(k); continue
                            nxt = n[end + 1:].split()[:WORDS]
                            if len(nxt) >= MIN_WORDS: cont.setdefault(k, []).append(" ".join(nxt))
                    if docs % 50_000 == 0:
                        print(f"  {src}: {docs:,} docs, {chars/1e9:.2f}G chars, {len(cont)} end-hits, "
                              f"{len(mid)} mid-hits, {(time.time()-t0)/60:.1f} min", flush=True)
            break
        except Exception as e:
            print(f"  {src}: read error {e!r}, retry {attempt + 1}", flush=True)
    print(f"{src}: done, {docs:,} docs, {chars/1e9:.2f}G chars, {len(cont)} end-hits, {len(mid)} mid-hits, "
          f"{(time.time()-t0)/60:.1f} min", flush=True)
    return src, {k: v[:20] for k, v in cont.items()}, mid

t0 = time.time()
with Pool(len(SOURCES)) as pool:
    results = pool.map(scan, SOURCES)
print(f"scanned in {(time.time()-t0)/60:.1f} min")

In [ ]:
# overlap statistics + hybrid submission
cont, mid, by_src = {}, set(), {}
for src, c, m in results:
    by_src[src] = dict(end_hits=len(c), mid_hits=len(m))
    for k, v in c.items(): cont.setdefault(k, []).extend(v)
    mid |= m
print("per source:", by_src)
print(f"prompts whose MIDDLE passage occurs in the training data: {len(mid)}/{len(test)} ({len(mid)/len(test):.1%})")
print(f"prompts whose END occurs with >= {MIN_WORDS} words after it: {len(cont)}/{len(test)} ({len(cont)/len(test):.1%})")

hybrid, used = {}, 0
for k in test:
    if k in cont:
        best, _ = Counter(cont[k]).most_common(1)[0]                 # most frequent continuation among duplicates
        w = best.split()
        if len(w) < WORDS: w += base[k]["text"].split()[: WORDS - len(w)]
        hybrid[k] = {"text": " ".join(w[:WORDS])}; used += 1
    else:
        hybrid[k] = {"text": base[k]["text"]}
agree = sum(len(set(v)) == 1 for v in cont.values())
print(f"replaced {used} predictions | prompts where all duplicates agree on the continuation: {agree}/{len(cont)}")

os.makedirs(os.path.join(OUT, "hybrid"), exist_ok=True)
p = os.path.join(OUT, "hybrid", "prediction.json"); json.dump(hybrid, open(p, "w", encoding="utf-8"), ensure_ascii=False)
zp = os.path.join(OUT, "submission_hybrid_retrieval.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(p, arcname="prediction.json")
with zipfile.ZipFile(zp) as z: assert set(json.loads(z.read("prediction.json"))) == set(test)
json.dump(dict(by_source=by_src, mid_hits=len(mid), end_hits=len(cont), replaced=used, duplicates_agree=agree),
          open(os.path.join(OUT, "overlap_stats.json"), "w"), indent=1)
k0 = next(iter(cont), None)
if k0: print("\nexample", k0, "\nPROMPT END:", norm(test[k0])[-200:], "\nRETRIEVED :", hybrid[k0]["text"][:300])
print("\nwrote", zp)